## Feature Engineering

In [1]:
# Cell 1 — Load all data
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
raw_dir       = Path.home() / "wc2026" / "data" / "raw"
processed_dir = Path.home() / "wc2026" / "data" / "processed"

# Load all datasets
results    = pd.read_csv(raw_dir       / "results.csv",            parse_dates=["date"])
fixture    = pd.read_csv(processed_dir / "fixtures_2026.csv",      parse_dates=["date"])
fifa_rank  = pd.read_csv(processed_dir / "fifa_rankings_2026.csv")
elo        = pd.read_csv(processed_dir / "elo_ratings_2026.csv")
squad      = pd.read_csv(processed_dir / "squad_values_2026.csv")
venue_heat = pd.read_csv(processed_dir / "venue_heat_index.csv")

print("Results:   ", results.shape)
print("Fixtures:  ", fixture.shape)
print("FIFA rank: ", fifa_rank.shape)
print("Elo:       ", elo.shape)
print("Squad:     ", squad.shape)
print("Venue heat:", venue_heat.shape)

print("\nAll data loaded!")

Results:    (49477, 9)
Fixtures:   (104, 16)
FIFA rank:  (48, 4)
Elo:        (336, 3)
Squad:      (48, 3)
Venue heat: (16, 4)

All data loaded!


In [2]:
# Quick sanity check
print("RESULTS:")
print(f"  Shape: {results.shape}")
print(f"  Date range: {results.date.min().date()} -> {results.date.max().date()}")

print("\nFIXTURES:")
print(f"  Shape: {fixture.shape}")
print(f"  Columns: {fixture.columns.tolist()}")

print("\nFIFA RANKINGS:")
print(f"  Shape: {fifa_rank.shape}")
print(f"  Columns: {fifa_rank.columns.tolist()}")

print("\nELO RATINGS:")
print(f"  Shape: {elo.shape}")
print(f"  Columns: {elo.columns.tolist()}")

print("\nSQUAD VALUES:")
print(f"  Shape: {squad.shape}")
print(f"  Columns: {squad.columns.tolist()}")

print("\nVENUE HEAT:")
print(f"  Shape: {venue_heat.shape}")
print(f"  Columns: {venue_heat.columns.tolist()}")

RESULTS:
  Shape: (49477, 9)
  Date range: 1872-11-30 -> 2026-06-27

FIXTURES:
  Shape: (104, 16)
  Columns: ['match_num', 'date', 'round', 'group', 'home_team', 'away_team', 'venue', 'home_score', 'away_score', 'stage', 'heat_index_scaled', 'home_heat_tolerance', 'away_heat_tolerance', 'neutral', 'home_advantage', 'away_advantage']

FIFA RANKINGS:
  Shape: (48, 4)
  Columns: ['fifa_rank', 'team', 'fifa_points', 'confederation']

ELO RATINGS:
  Shape: (336, 3)
  Columns: ['elo_rank', 'team', 'elo_rating']

SQUAD VALUES:
  Shape: (48, 3)
  Columns: ['value_rank', 'team', 'squad_value_eur_m']

VENUE HEAT:
  Shape: (16, 4)
  Columns: ['venue', 'temp_c', 'humidity', 'heat_index']


In [3]:
# Filter Elo to WC48 teams only
real_teams = sorted(pd.concat([
    fixture["home_team"],
    fixture["away_team"]
]).unique().tolist())
real_teams = [t for t in real_teams if not t.startswith(("L","W","1","2","3"))]

elo_wc = elo[elo["team"].isin(real_teams)].sort_values("elo_rating", ascending=False).reset_index(drop=True)

print(f"WC48 Elo ratings: {len(elo_wc)} teams")
print(elo_wc.head(10).to_string(index=False))

WC48 Elo ratings: 47 teams
 elo_rank        team  elo_rating
        1       Spain      2080.2
        2   Argentina      2073.6
        3      France      2017.1
        4    Portugal      1962.7
        5     England      1960.9
        6      Brazil      1956.1
        7    Colombia      1946.1
        8     Ecuador      1937.8
        9     Germany      1921.4
       10 Netherlands      1915.8


### Training Table Structure
One row per PLAYED match (from historical results).
For each match we need:
- The result (home goals, away goals) — this is what we PREDICT
- Features describing both teams BEFORE the match — this is what we TRAIN ON

In [4]:
# Build base training table from historical results
# Only played matches, sorted by date
training = results[results["home_score"].notna()].copy()
training = training.sort_values("date").reset_index(drop=True)

# Keep only relevent columns
training = training[[
    "date", "home_team", "away_team",
    "home_score", "away_score",
    "tournament", "neutral"
]]

# Convert score to integer
training["home_score"] = training["home_score"].astype(int)
training["away_score"] = training["away_score"].astype(int)

print("Training table shape:", training.shape)
print("\nSample:")
print(training.head(10).to_string(index=False))

Training table shape: (49405, 7)

Sample:
      date home_team away_team  home_score  away_score tournament  neutral
1872-11-30  Scotland   England           0           0   Friendly    False
1873-03-08   England  Scotland           4           2   Friendly    False
1874-03-07  Scotland   England           2           1   Friendly    False
1875-03-06   England  Scotland           2           2   Friendly    False
1876-03-04  Scotland   England           3           0   Friendly    False
1876-03-25  Scotland     Wales           4           0   Friendly    False
1877-03-03   England  Scotland           1           3   Friendly    False
1877-03-05     Wales  Scotland           0           2   Friendly    False
1878-03-02  Scotland   England           7           2   Friendly    False
1878-03-23  Scotland     Wales           9           0   Friendly    False


In [5]:
# adding match weight 
def get_match_weight(tournament):
    if tournament == "FIFA World Cup":
        return 1.0
    if tournament == "FIFA World Cup qualification":
        return 0.8
    if tournament in ["UEFA Euro", "Copa América",
                      "African Cup of Nations",
                      "AFC Asian Cup"]:
        return 0.9
    if "qualification" in tournament.lower():
        return 0.6
    if "Friendly" in tournament:
        return 0.3
   
    return 0.5
# Add column
training["match_weight"] = training["tournament"].apply(get_match_weight)

print("Match weight by tournament type:")
print(training.groupby("tournament")["match_weight"].first().sort_values(ascending=False).head(15).to_string())

Match weight by tournament type:
tournament
FIFA World Cup                             1.0
AFC Asian Cup                              0.9
Copa América                               0.9
African Cup of Nations                     0.9
UEFA Euro                                  0.9
FIFA World Cup qualification               0.8
ASEAN Championship qualification           0.6
AFC Asian Cup qualification                0.6
CONIFA World Football Cup qualification    0.6
COSAFA Cup qualification                   0.6
CONIFA World Cup qualification             0.6
Copa América qualification                 0.6
CONCACAF Nations League qualification      0.6
CFU Caribbean Cup qualification            0.6
CONCACAF Championship qualification        0.6


In [6]:
# Check African Cup of Nations tournament names
print(training[training["tournament"].str.contains("African", case=False)]["tournament"].unique())

<ArrowStringArray>
[              'African Cup of Nations',
             'African Friendship Games',
 'African Cup of Nations qualification',
                    'All-African Games',
                     'West African Cup',
 'Morocco, Capital of African Football']
Length: 6, dtype: str


## Add the Elo ratings to the training table. This is the most important feature.

In [7]:
# Add Elo ratings to training table
# We join the CURRENT Elo ratings onto the training table
# Later we will compute pre-match Elo properly (no leakage)

training = training.merge(
    elo[["team", "elo_rating"]].rename(columns={"team": "home_team",
                                               "elo_rating": "home_elo"}),
    on = "home_team",
    how = "left"
)

training = training.merge(
    elo[["team", "elo_rating"]].rename(columns={"team": "away_team",
                                               "elo_rating": "away_elo"}),
    on = "away_team",
    how = "left"
)

# Elo difference - positive means home team stronger
training["elo_diff"] = training["home_elo"] - training["away_elo"]

print("Missing Elo values:")
print(f"  home_elo: {training['home_elo'].isna().sum()}")
print(f"  away_elo: {training['away_elo'].isna().sum()}")

print("\nSample:")
print(training[["home_team","away_team","home_elo","away_elo","elo_diff"]].head(5).to_string(index=False))

Missing Elo values:
  home_elo: 635
  away_elo: 438

Sample:
home_team away_team  home_elo  away_elo  elo_diff
 Scotland   England    1773.0    1960.9    -187.9
  England  Scotland    1960.9    1773.0     187.9
 Scotland   England    1773.0    1960.9    -187.9
  England  Scotland    1960.9    1773.0     187.9
 Scotland   England    1773.0    1960.9    -187.9


In [8]:
# Check missing Elo teams
missing_home = training[training["home_elo"].isna()]["home_team"].unique()
missing_away = training[training["away_elo"].isna()]["away_team"].unique()

all_missing = set(missing_home) | set(missing_away)
print(f"Teams with no Elo rating: {len(all_missing)}")
print(sorted(all_missing)[:20])

Teams with no Elo rating: 2
['Bosnia and Herzegovina', 'United States']


### The results spine uses "Bosnia and Herzegovina" and "United States" but our Elo table has "Bosnia & Herzegovina" and "USA".

In [9]:
# Fix name mismatches in fifa_rank and squad

fifa_rank["team"] = fifa_rank["team"].replace({
    "USA":                  "United States",
    "Bosnia & Herzegovina": "Bosnia and Herzegovina"
})

squad["team"] = squad["team"].replace({
    "USA":                  "United States",
    "Bosnia & Herzegovina": "Bosnia and Herzegovina"
})

print("fifa_rank names fixed")
print("squad names fixed")

# Verify
print("\nUSA in fifa_rank:", "United States" in fifa_rank["team"].values)
print("Bosnia in fifa_rank:", "Bosnia and Herzegovina" in fifa_rank["team"].values)

fifa_rank names fixed
squad names fixed

USA in fifa_rank: True
Bosnia in fifa_rank: True


In [10]:
# Check training table
print("Training table shape:", training.shape)
print("\nColumns:", training.columns.tolist())
print("\nSample:")
print(training[["date","home_team","away_team","home_score","away_score",
                "match_weight","home_elo","away_elo","elo_diff"]].tail(5).to_string(index=False))

Training table shape: (49405, 11)

Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral', 'match_weight', 'home_elo', 'away_elo', 'elo_diff']

Sample:
      date   home_team  away_team  home_score  away_score  match_weight  home_elo  away_elo  elo_diff
2026-06-09    Ethiopia     Malawi           1           1           0.3    1416.4    1422.8      -6.4
2026-06-10     England Costa Rica           3           0           0.3    1960.9    1702.4     258.5
2026-06-10    Portugal    Nigeria           2           1           0.3    1962.7    1799.6     163.1
2026-06-10     Bolivia    Algeria           0           4           0.3    1667.3    1826.5    -159.2
2026-06-10 Afghanistan   Pakistan           0           2           0.5    1235.6    1176.2      59.4


### Add FIFA ranking points.

In [11]:
# Add FIFA ranking points

training = training.merge(
    fifa_rank[["team","fifa_points","fifa_rank"]].rename(
        columns={"team": "home_team",
                "fifa_points": "home_fifa_points",
                "fifa_rank": "home_fifa_rank"}),
    on = "home_team",
    how = "left"
)

training = training.merge(
    fifa_rank[["team","fifa_points","fifa_rank"]].rename(
        columns={"team": "away_team",
                "fifa_points": "away_fifa_points",
                "fifa_rank": "away_fifa_rank"}),
    on = "away_team",
    how = "left"
)

# Add FIFA poin difference
training["fifa_points_diff"] = training["home_fifa_points"] - training["away_fifa_points"]

print("Missing FIFA points:")
print(f"  home: {training['home_fifa_points'].isna().sum()}")
print(f"  away: {training['away_fifa_points'].isna().sum()}")

print("\nShape:", training.shape)

Missing FIFA points:
  home: 32020
  away: 33341

Shape: (49405, 16)


In [12]:
# Check missing for WC48 teams only
recent = training[training["date"]>="2018-01-01"]

missing_home = recent[recent["home_fifa_points"].isna()]["home_team"].unique()
missing_away = recent[recent["away_fifa_points"].isna()]["away_team"].unique()

all_missing = set(missing_home) | set(missing_away)
wc_missing = [t for t in all_missing if t in 
             pd.concat([fixture["home_team"], fixture["away_team"]]).unique()]

print(f"WC48 teams missing FIFA points since 2018: {len(wc_missing)}")
print(wc_missing if wc_missing else "None — all WC teams covered!")

WC48 teams missing FIFA points since 2018: 0
None — all WC teams covered!


### Merge Squad values from Transfermkt.

In [13]:
# Merge squad values

training = training.merge(
    squad[["team", "squad_value_eur_m"]].rename(
        columns = {"team": "home_team",
                  "squad_value_eur_m": "home_squad_value"}),
    on = "home_team",
    how = "left"
)

training = training.merge(
    squad[["team", "squad_value_eur_m"]].rename(
        columns = {"team": "away_team",
                  "squad_value_eur_m": "away_squad_value"}),
    on = "away_team",
    how = "left"
)

training["squad_value_diff"] = training["home_squad_value"] - training["away_squad_value"]

print("Shape:", training.shape)
print("Missing home_squad_value:", training["home_squad_value"].isna().sum())
print("Missing away_squad_value:", training["away_squad_value"].isna().sum())
print("\nColumns:", training.columns.tolist())


Shape: (49405, 19)
Missing home_squad_value: 32020
Missing away_squad_value: 33341

Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral', 'match_weight', 'home_elo', 'away_elo', 'elo_diff', 'home_fifa_points', 'home_fifa_rank', 'away_fifa_points', 'away_fifa_rank', 'fifa_points_diff', 'home_squad_value', 'away_squad_value', 'squad_value_diff']


In [14]:
# Final check and save training table
print("Training table shape:", training.shape)
print("\nDate range:", training.date.min().date(), "->", training.date.max().date())

# Check recent matches look good
print("\nSample — recent matches:")
print(training[training["date"] >= "2026-01-01"][
    ["date","home_team","away_team","home_score","away_score",
     "home_elo","away_elo","home_fifa_points","home_squad_value"]
].tail(5).to_string(index=False))

# Save
training.to_csv(processed_dir / "training_table.csv", index=False)
print("\nSaved!")

Training table shape: (49405, 19)

Date range: 1872-11-30 -> 2026-06-10

Sample — recent matches:
      date   home_team  away_team  home_score  away_score  home_elo  away_elo  home_fifa_points  home_squad_value
2026-06-09    Ethiopia     Malawi           1           1    1416.4    1422.8               NaN               NaN
2026-06-10     England Costa Rica           3           0    1960.9    1702.4           1825.97            1360.0
2026-06-10    Portugal    Nigeria           2           1    1962.7    1799.6           1763.83            1010.0
2026-06-10     Bolivia    Algeria           0           4    1667.3    1826.5               NaN               NaN
2026-06-10 Afghanistan   Pakistan           0           2    1235.6    1176.2               NaN               NaN

Saved!


In [15]:
# Training table summary

print("=" * 50)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 50)

print("\nTraining table: 49,405 matches")
print("\nFeatures built:")
print("  ✓ match_weight      — importance of each match")
print("  ✓ home_elo          — home team Elo rating")
print("  ✓ away_elo          — away team Elo rating")
print("  ✓ elo_diff          — Elo gap between teams")
print("  ✓ home_fifa_points  — home team FIFA points")
print("  ✓ away_fifa_points  — away team FIFA points")
print("  ✓ fifa_points_diff  — FIFA points gap")
print("  ✓ home_squad_value  — home squad value €M")
print("  ✓ away_squad_value  — away squad value €M")
print("  ✓ squad_value_diff  — squad value gap")

print("\nStill to add in next notebook:")
print("  — Recent form (rolling last N matches)")
print("  — Home advantage flag")
print("  — Heat stress (for WC26 fixtures only)")

print("\nNext: Notebook 3 — Poisson Model")

FEATURE ENGINEERING COMPLETE

Training table: 49,405 matches

Features built:
  ✓ match_weight      — importance of each match
  ✓ home_elo          — home team Elo rating
  ✓ away_elo          — away team Elo rating
  ✓ elo_diff          — Elo gap between teams
  ✓ home_fifa_points  — home team FIFA points
  ✓ away_fifa_points  — away team FIFA points
  ✓ fifa_points_diff  — FIFA points gap
  ✓ home_squad_value  — home squad value €M
  ✓ away_squad_value  — away squad value €M
  ✓ squad_value_diff  — squad value gap

Still to add in next notebook:
  — Recent form (rolling last N matches)
  — Home advantage flag
  — Heat stress (for WC26 fixtures only)

Next: Notebook 3 — Poisson Model


In [16]:
# Add home advantage flag
# True home = host nations in WC (USA, Mexico, Canada)
# Neutral = everyone else at WC
# For historical matches, use the neutral column directly

training["home_advantage"] = (~training["neutral"]).astype(int)


print("Home advantage distribution:")
print(training["home_advantage"].value_counts().to_string())
print("\nSample:")
print(training[["home_team","away_team","neutral","home_advantage"]].head(5).to_string(index=False))

Home advantage distribution:
home_advantage
1    36347
0    13058

Sample:
home_team away_team  neutral  home_advantage
 Scotland   England    False               1
  England  Scotland    False               1
 Scotland   England    False               1
  England  Scotland    False               1
 Scotland   England    False               1


In [17]:
# Add neutral column to fixture
hosts = ["United States", "Mexico", "Canada"]

fixture["neutral"] = ~fixture["home_team"].isin(hosts)
fixture["home_advantage"] = fixture["home_team"].isin(hosts).astype(int)

print("Host nation fixtures:")
print(fixture[fixture["home_advantage"] == 1][
    ["home_team","away_team","venue","home_advantage","neutral"]
].to_string(index=False))

print(f"\nTotal home advantage matches: {fixture['home_advantage'].sum()}")
print(f"Total neutral matches: {fixture['neutral'].sum()}")

Host nation fixtures:
    home_team            away_team                   venue  home_advantage  neutral
       Mexico         South Africa             Mexico City               1    False
       Mexico          South Korea   Guadalajara (Zapopan)               1    False
       Canada Bosnia & Herzegovina                 Toronto               1    False
       Canada                Qatar               Vancouver               1    False
United States             Paraguay Los Angeles (Inglewood)               1    False
United States            Australia                 Seattle               1    False

Total home advantage matches: 6
Total neutral matches: 98


In [18]:
# Check home advantage in WC26 fixtures

print("HOST nation fixtures (neutral should be False):")
hosts = ["United States", "Mexico", "Canada"]
host_fixtures = fixture[
    (fixture["home_team"].isin(hosts)) & 
    (fixture["stage"] == "Group")
][["home_team","away_team","venue","neutral"]]
print(host_fixtures.to_string(index=False))

print("\nNON-HOST group fixtures (neutral should be True):")
non_host = fixture[
    (~fixture["home_team"].isin(hosts)) & 
    (fixture["stage"] == "Group")
][["home_team","away_team","venue","neutral"]].head(5)
print(non_host.to_string(index=False))

HOST nation fixtures (neutral should be False):
    home_team            away_team                   venue  neutral
       Mexico         South Africa             Mexico City    False
       Mexico          South Korea   Guadalajara (Zapopan)    False
       Canada Bosnia & Herzegovina                 Toronto    False
       Canada                Qatar               Vancouver    False
United States             Paraguay Los Angeles (Inglewood)    False
United States            Australia                 Seattle    False

NON-HOST group fixtures (neutral should be True):
     home_team      away_team                                venue  neutral
   South Korea Czech Republic                Guadalajara (Zapopan)     True
Czech Republic   South Africa                              Atlanta     True
Czech Republic         Mexico                          Mexico City     True
  South Africa    South Korea                Monterrey (Guadalupe)     True
         Qatar    Switzerland San Francisco B

In [19]:
# Check all host fixtures
print("All host nation GROUP fixtures:")
print(fixture[
    (fixture["home_team"].isin(hosts)) & 
    (fixture["stage"] == "Group")
][["home_team","away_team","venue","neutral"]].to_string(index=False))

print("\nCount by host:")
print(fixture[fixture["home_team"].isin(hosts)].groupby("home_team").size())

All host nation GROUP fixtures:
    home_team            away_team                   venue  neutral
       Mexico         South Africa             Mexico City    False
       Mexico          South Korea   Guadalajara (Zapopan)    False
       Canada Bosnia & Herzegovina                 Toronto    False
       Canada                Qatar               Vancouver    False
United States             Paraguay Los Angeles (Inglewood)    False
United States            Australia                 Seattle    False

Count by host:
home_team
Canada           2
Mexico           2
United States    2
dtype: int64


In [20]:
# Check if hosts appear as away team
print("Host nations as AWAY team:")
print(fixture[
    (fixture["away_team"].isin(hosts)) & 
    (fixture["stage"] == "Group")
][["home_team","away_team","venue","neutral"]].to_string(index=False))

Host nations as AWAY team:
     home_team     away_team                   venue  neutral
Czech Republic        Mexico             Mexico City     True
   Switzerland        Canada               Vancouver     True
        Turkey United States Los Angeles (Inglewood)     True


In [21]:
# Fix home advantage - hosts get it whether home or away
hosts = ["USA", "Mexico", "Canada"]

fixture["neutral"] = ~(
    fixture["home_team"].isin(hosts) | 
    fixture["away_team"].isin(hosts)
)

fixture["home_advantage"] = fixture["home_team"].isin(hosts).astype(int)
fixture["away_advantage"] = fixture["away_team"].isin(hosts).astype(int)

print("Host fixtures (home or away):")
print(fixture[
    (fixture["home_team"].isin(hosts) | fixture["away_team"].isin(hosts)) &
    (fixture["stage"] == "Group")
][["home_team","away_team","venue","neutral","home_advantage","away_advantage"]].to_string(index=False))

print("\nCount:")
print(f"USA games:    {fixture[fixture['home_team'].eq('USA') | fixture['away_team'].eq('USA')]['stage'].eq('Group').sum()}")
print(f"Mexico games: {fixture[fixture['home_team'].eq('Mexico') | fixture['away_team'].eq('Mexico')]['stage'].eq('Group').sum()}")
print(f"Canada games: {fixture[fixture['home_team'].eq('Canada') | fixture['away_team'].eq('Canada')]['stage'].eq('Group').sum()}")

Host fixtures (home or away):
     home_team            away_team                 venue  neutral  home_advantage  away_advantage
        Mexico         South Africa           Mexico City    False               1               0
        Mexico          South Korea Guadalajara (Zapopan)    False               1               0
Czech Republic               Mexico           Mexico City    False               0               1
        Canada Bosnia & Herzegovina               Toronto    False               1               0
        Canada                Qatar             Vancouver    False               1               0
   Switzerland               Canada             Vancouver    False               0               1

Count:
USA games:    0
Mexico games: 3
Canada games: 3


In [22]:
# Fix USA name consistently across all dataframes
fixture["home_team"] = fixture["home_team"].replace({"USA": "United States"})
fixture["away_team"] = fixture["away_team"].replace({"USA": "United States"})

# Verify
print("USA still in fixture:", "USA" in fixture["home_team"].values or "USA" in fixture["away_team"].values)
print("United States in fixture:", "United States" in fixture["home_team"].values or "United States" in fixture["away_team"].values)

# Save
fixture.to_csv(processed_dir / "fixtures_2026.csv", index=False)
print("Saved!")

USA still in fixture: False
United States in fixture: True
Saved!


In [23]:
# Save training table
training.to_csv(processed_dir / "training_table.csv", index=False)

print("Saved!")
print("\nTraining table shape:", training.shape)
print("Columns:", training.columns.tolist())

Saved!

Training table shape: (49405, 20)
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral', 'match_weight', 'home_elo', 'away_elo', 'elo_diff', 'home_fifa_points', 'home_fifa_rank', 'away_fifa_points', 'away_fifa_rank', 'fifa_points_diff', 'home_squad_value', 'away_squad_value', 'squad_value_diff', 'home_advantage']


##  Recent form — last 5 COMPETITIVE matches, weighted by match importance.

In [24]:
#  Recent form — last 5 COMPETITIVE matches, weighted by match importance

def get_recent_form(df, team, match_date, n=5):
    """
    Get weighted win rate for a team in their last n competitive matches.
    Excludes friendlies — they don't reflect true team strength.
    Weights results by match importance (WC > qualifier > other).
    """
    mask = (
        ((df["home_team"]== team) | (df["away_team"]==team)) &
        (df["date"]< match_date) &
        (df["match_weight"]>0.3)    # exclude friendly
    )

    recent = df[mask].tail(n)

    if len(recent) ==0:
        return np.nan

    weighted_wins = 0
    total_weight = 0

    for _, row in recent.iterrows():
        if row["home_team"] == team:
            won = row["home_score"] > row["away_score"]
        else:
            won = row["away_score"] > row["home_score"]

        weighted_wins += row["match_weight"] * int(won)
        total_weight += row["match_weight"]
    
    return weighted_wins / total_weight

# Test
print("Testing weighted competitve form:")
print(f"England: {get_recent_form(training, 'England', pd.Timestamp('2026-01-01')):.2f}")
print(f"France: {get_recent_form(training, 'France', pd.Timestamp('2026-01-01')):.2f}")
print(f"Brazil: {get_recent_form(training, 'Brazil', pd.Timestamp('2026-01-01')):.2f}")
print(f"Argentina: {get_recent_form(training, 'Argentina', pd.Timestamp('2026-01-01')):.2f}")
print(f"Haiti: {get_recent_form(training, 'Haiti', pd.Timestamp('2026-01-01')):.2f}")

Testing weighted competitve form:
England: 1.00
France: 0.80
Brazil: 0.43
Argentina: 0.60
Haiti: 0.60


In [25]:
# Apply to full training table

print("Computing recent form for all matches:")

training["home_form"] = [
    get_recent_form(training, row.home_team, row.date)
    for row in training.itertuples()
]

training["away_form"] = [
    get_recent_form(training, row.away_team, row.date)
    for row in training.itertuples()
]

print("Done!")
print("Missing home_form", training["home_form"].isna().sum())
print("Missing away_form", training["away_form"].isna().sum())


Computing recent form for all matches:
Done!
Missing home_form 1336
Missing away_form 1302


In [26]:
# Quick check on recent form

print("Sample recent form for recent matches:")
print(training[training["date"]>="2025-01-01"][
     ["date","home_team","away_team","home_form","away_form"]
].tail(10).to_string(index=False))

Sample recent form for recent matches:
      date   home_team                away_team  home_form  away_form
2026-06-09  Azerbaijan               San Marino   0.147059   0.000000
2026-06-09     Armenia                  Moldova   0.200000   0.000000
2026-06-09   Argentina                  Iceland   0.600000   0.200000
2026-06-09      Angola Central African Republic   0.000000   0.200000
2026-06-09        Iraq                Venezuela   0.642857   0.147059
2026-06-09    Ethiopia                   Malawi   0.400000   0.432432
2026-06-10     England               Costa Rica   1.000000   0.200000
2026-06-10    Portugal                  Nigeria   0.600000   0.513514
2026-06-10     Bolivia                  Algeria   0.432432   0.800000
2026-06-10 Afghanistan                 Pakistan   0.178571   0.357143


In [27]:
# Save final training table
training.to_csv(processed_dir / "training_table.csv", index=False)

print("Saved!")
print("\nFinal training table shape:", training.shape)
print("\nAll features:")
for col in training.columns:
    missing = training[col].isna().sum()
    print(f"  {col:<25} missing: {missing:,}")

Saved!

Final training table shape: (49405, 22)

All features:
  date                      missing: 0
  home_team                 missing: 0
  away_team                 missing: 0
  home_score                missing: 0
  away_score                missing: 0
  tournament                missing: 0
  neutral                   missing: 0
  match_weight              missing: 0
  home_elo                  missing: 635
  away_elo                  missing: 438
  elo_diff                  missing: 1,070
  home_fifa_points          missing: 32,020
  home_fifa_rank            missing: 32,020
  away_fifa_points          missing: 33,341
  away_fifa_rank            missing: 33,341
  fifa_points_diff          missing: 41,878
  home_squad_value          missing: 32,020
  away_squad_value          missing: 33,341
  squad_value_diff          missing: 41,878
  home_advantage            missing: 0
  home_form                 missing: 1,336
  away_form                 missing: 1,302


## Merging updated fifa ranking - all teams

In [31]:
# Check name coverage against training table
fifa_all = pd.read_csv(raw_dir / "fifa_rankings_all_teams.csv")

training_teams = pd.concat([training["home_team"], training["away_team"]]).unique()
missing = [t for t in training_teams if t not in fifa_all["team"].values]

print(f"Teams in training not in FIFA rankings: {len(missing)}")
print(f"Teams covered: {len(training_teams) - len(missing)}/{len(training_teams)}")
print("\nMissing sample:")
for t in sorted(missing)[:20]:
    print(f"  {t}")

Teams in training not in FIFA rankings: 135
Teams covered: 201/336

Missing sample:
  Abkhazia
  Alderney
  Ambazonia
  Andalusia
  Arameans Suryoye
  Artsakh
  Asturias
  Aymara
  Barawa
  Basque Country
  Biafra
  Bonaire
  Bosnia and Herzegovina
  Brittany
  Canary Islands
  Cascadia
  Catalonia
  Central Spain
  Chagos Islands
  Chameria


In [32]:
# Check which missing teams are actual WC teams
wc_missing = [t for t in missing if t in 
              pd.concat([fixture["home_team"], fixture["away_team"]]).unique()]

print(f"WC48 teams missing from FIFA rankings: {len(wc_missing)}")
print(wc_missing if wc_missing else "None — all WC teams covered!")

# Check name mismatches only
print(f"\nLikely name mismatches (real countries):")
noise = ["Abkhazia","Alderney","Ambazonia","Andalusia","Arameans",
         "Artsakh","Asturias","Aymara","Barawa","Basque","Biafra",
         "Bonaire","Brittany","Canary","Cascadia","Catalonia"]
real_missing = [t for t in missing if not any(n in t for n in noise)]
for t in sorted(real_missing)[:20]:
    print(f"  {t}")

WC48 teams missing from FIFA rankings: 0
None — all WC teams covered!

Likely name mismatches (real countries):
  Bosnia and Herzegovina
  Central Spain
  Chagos Islands
  Chameria
  Chechnya
  China
  Cilento
  Corsica
  County of Nice
  Crimea
  Czechoslovakia
  Darfur
  Donetsk PR
  Délvidék
  East Turkestan
  Elba Island
  Ellan Vannin
  Falkland Islands
  Felvidék
  Franconia


## Merging updated FIFA rankings 

In [34]:
# Load full FIFA rankings
fifa_all = pd.read_csv(raw_dir / "fifa_rankings_all_teams.csv")

# Merge onto training table
training = training.drop(columns=["home_fifa_points", "home_fifa_rank", 
                                   "away_fifa_points", "away_fifa_rank",
                                   "fifa_points_diff"])

training = training.merge(
    fifa_all[["team","fifa_points","rank"]].rename(
        columns={"team":"home_team","fifa_points":"home_fifa_points","rank":"home_fifa_rank"}),
    on="home_team", how="left"
)

training = training.merge(
    fifa_all[["team","fifa_points","rank"]].rename(
        columns={"team":"away_team","fifa_points":"away_fifa_points","rank":"away_fifa_rank"}),
    on="away_team", how="left"
)

training["fifa_points_diff"] = training["home_fifa_points"] - training["away_fifa_points"]

print("Missing FIFA points:")
print(f"  home: {training['home_fifa_points'].isna().sum()}")
print(f"  away: {training['away_fifa_points'].isna().sum()}")
print("\nShape:", training.shape)

Missing FIFA points:
  home: 3768
  away: 4280

Shape: (49603, 22)


In [35]:
# Save updated training table
training.to_csv(processed_dir / "training_table.csv", index=False)

print("Saved!")
print("\nMissing values summary:")
print(training.isnull().sum()[training.isnull().sum() > 0].to_string())

Saved!

Missing values summary:
home_elo              638
away_elo              443
elo_diff             1078
home_squad_value    32198
away_squad_value    33516
squad_value_diff    42076
home_form            1337
away_form            1304
home_fifa_points     3768
home_fifa_rank       3768
away_fifa_points     4280
away_fifa_rank       4280
fifa_points_diff     6629


### Redo Merging Training table and Squad value

In [5]:
import pandas as pd
from pathlib import Path

processed_dir = Path.home() / "wc2026" / "data" / "processed"
raw_dir       = Path.home() / "wc2026" / "data" / "raw"

# Load trainig table and new squad values
training = pd.read_csv(processed_dir / "training_table.csv", parse_dates=['date'])
squad_new = pd.read_csv(raw_dir / "squad_values_all_teams.csv")

print("Training shape:", training.shape)
print("Squad teams:", len(squad_new))
print("Current missing squad values:", training["home_squad_value"].isna().sum())

Training shape: (49603, 22)
Squad teams: 209
Current missing squad values: 32198


In [6]:
# Drop old squad value columns
training = training.drop(columns=["home_squad_value","away_squad_value","squad_value_diff"])

# Merge new squad values - home team
training = training.merge(
    squad_new[["team", "squad_value_eur_m"]].rename(
        columns={"team": "home_team", "squad_value_eur_m": "home_squad_value"}),
    on = "home_team",
    how = "left"
)

# Merge new squad values - away team
training = training.merge(
    squad_new[["team", "squad_value_eur_m"]].rename(
        columns={"team": "away_team", "squad_value_eur_m": "away_squad_value"}),
    on = "away_team",
    how = "left"
)

training["squad_value_diff"] = training["home_squad_value"] - training["away_squad_value"]

print("Missing home_squad_value:", training["home_squad_value"].isna().sum())
print("Missing away_squad_value:", training["away_squad_value"].isna().sum())
print("Shape:", training.shape)

Missing home_squad_value: 2704
Missing away_squad_value: 3235
Shape: (49603, 22)


In [7]:
# Save updated training table
training.to_csv(processed_dir / "training_table.csv", index=False)

print("Saved!")
print("\nMissing values summary:")
print(training.isnull().sum()[training.isnull().sum() > 0].to_string())

Saved!

Missing values summary:
home_elo             638
away_elo             443
elo_diff            1078
home_form           1337
away_form           1304
home_fifa_points    3768
home_fifa_rank      3768
away_fifa_points    4280
away_fifa_rank      4280
fifa_points_diff    6629
home_squad_value    2704
away_squad_value    3235
squad_value_diff    4783
